# 03b. TabPFN, on Colab

The tabular foundation model of the comparison. TabPFN needs a GPU to be practical, so it lives here
rather than in `03_models.ipynb`.

Like the other two models, it predicts **the two decisions separately and multiplies them** rather
than predicting the match directly. Same folds, same target, so the three are comparable.

**How to run it**

1. run `03_models.ipynb` locally first, it writes `data/processed/model_table.csv`;
1. open this notebook in Colab and set *Runtime → Change runtime type → T4 GPU*;
2. run the cells; the upload cell asks for `data/processed/model_table.csv`;
3. the last cell downloads `tabpfn_oof.csv`. Put it in `data/processed/` and re-run the last cell of
   `03_models.ipynb` to merge it in.

The fold numbers travel inside the file, so TabPFN is scored on exactly the same split as the
logistic regression and XGBoost. Nothing here recomputes a split.

In [ ]:
!pip install tabpfn -q

In [ ]:
import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU. Runtime -> Change runtime type -> T4 GPU, or expect about ten minutes a fold.")

## 1. The data

In Colab this opens a file picker: choose `model_table.csv`. Outside Colab it reads the file
from the repository, so the notebook can also be run locally if the GPU is not available.

In [ ]:
import numpy as np
import pandas as pd

try:
    from google.colab import files
    uploaded = files.upload()
    table = pd.read_csv(next(iter(uploaded)))
except ImportError:
    table = pd.read_csv("data/processed/model_table.csv")

print(table.shape, "| folds:", sorted(table.fold.unique()))

## 2. One row per decision

A match is she says yes and he says yes. Each pair becomes two rows, one per decider, who becomes
`self_` while the partner becomes `other_`, and the two predicted probabilities are multiplied back
into a match probability at the end.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

TARGETS = ["pair", "wave", "match", "fold", "her_dec", "his_dec"]
SIDE_CATEGORICAL = ["self_field", "self_race", "self_goal",
                    "other_field", "other_race", "other_goal"]

def decision_view(tab, decider):
    mine, theirs = ("her", "his") if decider == "her" else ("his", "her")
    view = tab.drop(columns=TARGETS).rename(
        columns=lambda c: c.replace(f"{mine}_", "self_").replace(f"{theirs}_", "other_"))
    view["agediff"] = view.self_age - view.other_age   # directional: follows whoever is deciding
    view["female"] = int(decider == "her")
    return view

def preparation():
    encode = ColumnTransformer(
        [("categories", OneHotEncoder(min_frequency=0.03, handle_unknown="infrequent_if_exist",
                                      sparse_output=False), SIDE_CATEGORICAL)],
        remainder="passthrough")
    return make_pipeline(encode, SimpleImputer(strategy="median"))

hers, his = decision_view(table, "her"), decision_view(table, "his")
y = table.match
print(hers.shape, "rows per side ->", 2 * len(hers), "decisions")

## 3. Fit and predict, fold by fold

TabPFN does not train in the usual sense: it carries the training rows and makes its predictions in a
single forward pass, so a fold takes seconds on a GPU. `n_estimators` is how many internal ensemble
members it averages. Note that a training fold now holds about 6,700 decisions rather than 3,300
pairs, which is still well inside what TabPFN handles.

Two settings worth knowing about, both tested:

- `ignore_pretraining_limits=True` is only needed off the GPU. TabPFN refuses to run on a CPU with
  more than 1,000 rows, and we have 3,337 per training fold. On a GPU the flag changes nothing; on a
  CPU it makes the notebook run, slowly: count roughly ten minutes a fold.
- if the GPU runs out of memory, lower `n_estimators` to 4 or 2 before anything else.

In [ ]:
import time
from tabpfn import TabPFNClassifier

predictions = pd.Series(np.nan, index=table.index)

for k in sorted(table.fold.unique()):
    train, test = table.fold != k, table.fold == k
    prep = preparation()

    started = time.time()
    both_sides = pd.concat([hers[train], his[train]])
    decisions = pd.concat([table.her_dec[train], table.his_dec[train]])

    model = TabPFNClassifier(n_estimators=8, random_state=0, ignore_pretraining_limits=True)
    model.fit(prep.fit_transform(both_sides), decisions)

    predictions[test] = (model.predict_proba(prep.transform(hers[test]))[:, 1]
                         * model.predict_proba(prep.transform(his[test]))[:, 1])
    print(f"fold {k}: {len(both_sides)} decisions / {test.sum()} pairs, {time.time() - started:.0f}s")

print("missing predictions:", int(predictions.isna().sum()))

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

print("ROC-AUC:", round(roc_auc_score(y, predictions), 3))
print("PR-AUC: ", round(average_precision_score(y, predictions), 3))

## 4. Send it back

`tabpfn_oof.csv` holds one row per pair, keyed on `pair` so `03_models.ipynb` can join it to the
other two models' predictions.

In [ ]:
out = pd.DataFrame({"pair": table.pair, "tabpfn": predictions.to_numpy()})
out.to_csv("tabpfn_oof.csv", index=False)

try:
    from google.colab import files
    files.download("tabpfn_oof.csv")
except ImportError:
    print("saved tabpfn_oof.csv next to this notebook")

out.head(3)